# 02 · Modelos, validación y MLflow

Se comparan **regresión lineal, KNN, Random Forest, XGBoost y SVM**, además de **MLP base y tres experimentos**. Son implementaciones oficiales de sklearn/XGBoost, no algoritmos manuales.

Métrica principal: MAE en euros de renta municipal. Un error de 50 EUR afecta a la referencia que recibe el servicio de orientación, pero no implica que una persona pueda o no recibir una ayuda. También registramos RMSE, R², tiempo y métricas de entrenamiento.

In [1]:
from pathlib import Path
import os, sys, json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'AGENTS.md').exists())
sys.path.insert(0, str(ROOT))
OUT = Path(os.getenv('RETO1_ARTIFACT_DIR', str(ROOT / 'entrega_ml')))
from src.ml_workflow import train_official
from src.mlflow_tracker import MLflowTracker
if not (OUT / 'models/selection.json').exists():
    train_official(ROOT / 'data/capturas_oficiales', OUT)
else:
    print('Selección ya cerrada: se consultan experimentos existentes, sin reentrenar tras mirar test.')

Selección ya cerrada: se consultan experimentos existentes, sin reentrenar tras mirar test.


## Preprocesado y validación

Todos los regresores reciben el mismo `ColumnTransformer`: imputación por mediana y `StandardScaler` para numéricas; imputación y `OneHotEncoder` para categóricas. El pipeline completo queda guardado. La variable objetivo se estandariza con `TransformedTargetRegressor` y las predicciones vuelven a euros.

`GridSearchCV` usa ventanas temporales crecientes sobre train; un trimestre completo pertenece al mismo lado de cada corte. Se comparan vecinos en KNN, hojas mínimas en RF, profundidad en XGBoost y C en SVM. La validación 2024 elige el ganador. En empates dentro del 1% se prefiere regresión lineal. El test no interviene.

In [2]:
tracker = MLflowTracker(tracking_dir=OUT / 'mlruns')
runs = tracker.results()
display(runs[['run_id', 'tags.mlflow.runName', 'metrics.train_mae', 'metrics.val_mae', 'metrics.val_rmse', 'metrics.val_r2', 'metrics.tiempo_s']])
comparativa = pd.read_csv(OUT / 'resultados/comparativa_mlflow.csv')
display(comparativa)
seleccion = json.loads((OUT / 'models/selection.json').read_text())
display(seleccion)

,run_id,tags.mlflow.runName,metrics.train_mae,metrics.val_mae,metrics.val_rmse,metrics.val_r2,metrics.tiempo_s
0,68f3d500852a4d05b0f08f1de0984ec9,subida_con_SMOTE,NaN,NaN,NaN,NaN,0.008792
1,b7a73d4817d04568a184867c45996f10,subida_sin_SMOTE,NaN,NaN,NaN,NaN,0.006821
2,fd46a2b43de64a8bb0dea950b5a995fb,MLP_regularizada,1.477709e+01,25.417348,34.027333,0.928162,0.125040
3,3b9c14be1e5d42dc8275ab9315816b15,MLP_tanh,1.697235e+01,35.159188,50.787581,0.839965,0.139082
4,bcc44862768c4488ac8a04b41ece5efa,MLP_2_capas,1.588002e+01,24.197045,31.859715,0.937023,0.079891
5,02aec059022447a8861f631a2a6e7709,MLP_base,1.422348e+01,25.779904,34.375520,0.926684,0.125996
6,da581b079c4146a38fc304d473eaf5c2,SVM,1.060007e+01,37.862969,55.555705,0.808505,0.185577
7,ddf66baf250d4ae9a994a11bb492ed58,XGBoost,1.164371e+01,26.523390,37.051572,0.914825,0.249365
8,e52db9cb96f14384b6ffbc3159d487f3,Random_Forest,5.921718e+00,25.150672,34.761965,0.925026,1.846183
9,d513ab722c6f406495acdce2820b3b06,KNN,6.589755e-07,51.519593,60.891214,0.769957,0.091685


,modelo,run_id,tiempo_s,train_mae,train_rmse,train_r2,val_mae,val_rmse,val_r2,avisos_entrenamiento
0,MLP_2_capas,bcc44862768c4488ac8a04b41ece5efa,0.079891,1.588002e+01,22.352578,0.955787,24.197045,31.859715,0.937023,0.0
1,Random_Forest,e52db9cb96f14384b6ffbc3159d487f3,1.846183,5.921718e+00,9.095977,0.992679,25.150672,34.761965,0.925026,0.0
2,MLP_regularizada,fd46a2b43de64a8bb0dea950b5a995fb,0.125040,1.477709e+01,21.222882,0.960143,25.417348,34.027333,0.928162,0.0
3,MLP_base,02aec059022447a8861f631a2a6e7709,0.125996,1.422348e+01,19.838498,0.965173,25.779904,34.375520,0.926684,0.0
4,XGBoost,ddf66baf250d4ae9a994a11bb492ed58,0.249365,1.164371e+01,16.312388,0.976453,26.523390,37.051572,0.914825,0.0
5,Regresion_lineal,a112a6b5f8b849609396fe6a83ed22a1,0.225787,1.276370e+01,16.671928,0.975404,31.166496,40.583652,0.897812,0.0
6,MLP_tanh,3b9c14be1e5d42dc8275ab9315816b15,0.139082,1.697235e+01,23.440417,0.951379,35.159188,50.787581,0.839965,0.0
7,SVM,da581b079c4146a38fc304d473eaf5c2,0.185577,1.060007e+01,16.586481,0.975655,37.862969,55.555705,0.808505,0.0
8,KNN,d513ab722c6f406495acdce2820b3b06,0.091685,6.589755e-07,0.000003,1.000000,51.519593,60.891214,0.769957,0.0


{'modelo': 'MLP_2_capas',
 'criterio': 'MAE validacion; empate del 1% a favor de regresion lineal',
 'mae_val': 24.197045199402613,
 'baseline_persistencia_val': {'mae': 24.626666666666672,
  'rmse': 34.79208683977821,
  'r2': 0.9248964905926897},
 'sha256_modelo': 'd6f05d11fa6002de53ecaff671b697bd901f63f946b87cc7484701aea76c210a',
 'sha256_test': 'f1520fc0580fee119b031252538961e5d62eee68db6a0783c3a5384bc5a65011',
 'features': ['anio',
  'trimestre',
  'renta_lag1',
  'renta_lag4',
  'contratos_lag1',
  'codigo_municipio',
  'provincia'],
 'objetivo': 'renta_media_eur',
 'municipios': ['01059',
  '20030',
  '20040',
  '20045',
  '20055',
  '20067',
  '20069',
  '20071',
  '20079',
  '48013',
  '48015',
  '48020',
  '48027',
  '48044',
  '48054',
  '48078',
  '48082',
  '48084',
  '48902'],
 'max_periodo_entrenamiento': 8095,
 'max_periodo_validacion': 8099,
 'test_evaluado': False,
 'unidad_prediccion': 'renta media municipal trimestral, no precio de un anuncio individual'}

## Experimentos MLP y justificación

Base: una capa de 32 neuronas ReLU. Experimento 1: (64, 32), para comprobar si mayor capacidad mejora la validación. Experimento 2: 32 neuronas y tanh, cambiando activación. Experimento 3: 32 neuronas y alpha=0.01, aumentando regularización. Se mantiene semilla 42 y un máximo de 1.000 iteraciones.

Los avisos de entrenamiento quedan contabilizados; un warning de convergencia no se oculta ni equivale a un resultado validado. Hay que contrastar train/validación para detectar sobreajuste. La renta del trimestre anterior es un baseline obligatorio, calculado donde el retardo existe.

## SMOTE: ámbito correcto

SMOTE no se aplica a una renta continua. Como ampliación se compara una clasificación auxiliar de **subida trimestral observada** con y sin SMOTE, usando solo variables numéricas escaladas para evitar categorías fraccionarias. Se balancea exclusivamente train; validación mantiene su distribución. La matriz de confusión, precisión, recall y F1 proceden de validación. Este experimento no puntúa la viabilidad de personas ni determina la ayuda.

In [3]:
display(json.loads((OUT / 'resultados/comparacion_smote.json').read_text()))

{'objetivo': 'sube_renta_respecto_trimestre_anterior',
 'clases_train': {'1': 357, '0': 219},
 'datos': 'EMAL observado; solo numericas; no solicitantes ni elegibilidad',
 'modelos': {'subida_sin_SMOTE': {'run_id': 'b7a73d4817d04568a184867c45996f10',
   'accuracy': 0.7066666666666667,
   'precision': 0.7123287671232876,
   'recall': 0.9811320754716981,
   'f1': 0.8253968253968254,
   'matriz_confusion': {'TN': 1, 'FP': 21, 'FN': 1, 'TP': 52},
   'roc_auc': 0.6380789022298456,
   'tiempo_s': 0.006821005000006153},
  'subida_con_SMOTE': {'run_id': '68f3d500852a4d05b0f08f1de0984ec9',
   'accuracy': 0.7066666666666667,
   'precision': 0.7313432835820896,
   'recall': 0.9245283018867925,
   'f1': 0.8166666666666667,
   'matriz_confusion': {'TN': 4, 'FP': 18, 'FN': 4, 'TP': 49},
   'roc_auc': 0.6680960548885078,
   'tiempo_s': 0.008791725999998334}},
 'estado': 'comparacion_real_en_validacion_sin_tocar_test'}

## Evaluación final

El modelo elegido se conserva en `models/best_model.joblib`, incluyendo preprocesamiento. La pestaña **Test reservado** de Streamlit ejecuta una única evaluación y conserva sus huellas y resultados. Este notebook no abre ni puntúa ese conjunto. Las capturas deben corresponder al MLflow y a la app realmente ejecutados; no se inventan pantallas ni cifras.